# Import libraries

In [1]:
import pandas as pd
import time
import os
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv()

True

# LLMs

In [2]:
class DefaultResponse(BaseModel):
    response: str

In [3]:
class MedicalAgentClient:
    def __init__(self, base_url: str = None):
        self.__base_url = base_url or os.getenv("AGENT_URL", "http://localhost:8003")
        self.__ask_url = f"{self.__base_url}/ask"

    def ask(self, question: str, timeout: int = 3000) -> dict:
        response = requests.post(
            self.__ask_url,
            json={"question": question},
            headers={"Content-Type": "application/json"},
            timeout=timeout,
        )
        response.raise_for_status()
        return response.json()

    def health_check(self) -> bool:
        try:
            r = requests.get(f"{self.__base_url}/health", timeout=10)
            return r.status_code == 200
        except Exception:
            return False

In [4]:
class GPT:
    def __init__(self):
        self.__client = OpenAI()

    def generate_response(self, system_prompt, user_prompt, output_schema=DefaultResponse, model_name="gpt-4.1"):
        response = self.__client.responses.parse(
            model=model_name,
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.0,
            text_format=output_schema,
        )

        return response.output_parsed


# Prompts

In [5]:
class FactCheck(BaseModel):
    statement: str = Field(
        description="Pojedyncze, wyizolowane zdanie lub stwierdzenie wyciągnięte z uzasadnienia."
    )
    is_true: bool = Field(
        description="Czy to konkretne stwierdzenie jest w 100% zgodne z prawdą historyczną, naukową lub logiczną?"
    )
    error_explanation: str | None = Field(
        description="Jeśli is_true to False, napisz krótko, na czym polega błąd. Jeśli True, zostaw null/puste."
    )

class EvaluationResponse(BaseModel):
    extracted_facts: list[FactCheck] = Field(
        description="Rozbij całe uzasadnienie na pojedyncze, atomowe fakty i oceń prawdziwość każdego z nich z osobna."
    )
    is_answer_correct: bool = Field(
        description=(
            "Zwróć True TYLKO wtedy, gdy oceniana odpowiedź końcowa jest merytorycznie "
            "w 100% zgodna z wzorcem. Zwróć False, jeśli występuje jakiekolwiek odstępstwo, "
            "błąd lub jeśli odpowiedź jest tylko częściowo poprawna."
        )
    )
    is_reasoning_correct: bool = Field(
        description=(
            "Zwróć True TYLKO ORAZ WYŁĄCZNIE, jeśli uzasadnienie jest absolutnie bezbłędne "
            "zarówno pod kątem logicznym, jak i FAKTOGRAFICZNYM. Wszystkie użyte w nim daty, liczby, "
            "nazwy i fakty muszą być w 100% prawdziwe. "
            "Zwróć BEZWZGLĘDNIE False, jeśli w uzasadnieniu pojawi się JAKIKOLWIEK błąd rzeczowy "
            "(np. błędny rok), fałszywe założenie, halucynacja lub zgadywanie, nawet jeśli "
            "końcowy wynik jest poprawny. "
            "Jedyny wyjątek, kiedy zwracasz True pomimo błędu, to omyłka polegająca na złym przepisaniu "
            "poprawnego wyniku z uzasadnienia do ostatecznej odpowiedzi."
        )
    )

calibrated_reasoning_prompt = """
Jesteś niezwykle surowym ewaluatorem faktów i logiki.
Twoim zadaniem jest ocena uzasadnienia i odpowiedzi końcowej.

ZASADA ZERO TOLERANCJI: Wymagam absolutnej precyzji. Najpierw musisz rozbić uzasadnienie na pojedyncze fakty i zweryfikować każdy z nich. Wystąpienie choćby jednego błędnego faktu (np. fałszywa data, zła liczba) bezwzględnie dyskwalifikuje całe uzasadnienie.

### DANE WEJŚCIOWE:
Pytanie: {question}
Wzorzec (Poprawna odpowiedź): {correct_answer}
Wygenerowane uzasadnienie: {reason}

### ZADANIE:
Wypełnij schemat. Pamiętaj: najpierw rozbij tekst na fakty i bezlitośnie je zweryfikuj. Jeśli jakikolwiek wyciągnięty fakt jest błędny, główna flaga 'is_reasoning_correct' musi wynosić False.
"""

# Define metrics

In [6]:
def measure_response_time(function, *args, **kwargs):
    start = time.perf_counter()
    result = function(*args, **kwargs)
    end = time.perf_counter()

    return end - start, result

In [7]:
def count_words(text):
    text = text.split()

    return len(text)

In [8]:
def calibrated_reasoning(question: str, correct_answer: str, reason: str) -> EvaluationResponse:
    gpt = GPT()

    prompt = calibrated_reasoning_prompt.format(
        question=question,
        correct_answer=correct_answer,
        reason=reason,
    )

    response = gpt.generate_response(prompt, "", EvaluationResponse, "gpt-4.1")
    return response

# Verify agent is running

In [9]:
agent = MedicalAgentClient()

if agent.health_check():
    print("Agent works.")
else:
    print("No response.")

Agent works.


# Generate answers (Agent)

In [2]:
data = pd.read_json("data/lek_pl_sample.json")
data.drop(["edition", "year", "season", "question_id"], axis=1, inplace=True)
data.reset_index(inplace=True, drop=True)
data.rename(columns={"question_w_options": "question"}, inplace=True)

OUTPUT_FILE = "data/agent_results.jsonl"

In [10]:
def save_result(record, file_path=OUTPUT_FILE):
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


START_FROM = 305

for i in range(START_FROM, len(data)):
    question = data.iloc[i]["question"]
    correct_answer = data.iloc[i]["answer"]

    try:
        response_time, agent_response = measure_response_time(
            agent.ask,
            question
        )

        text = agent_response.get("final_response") or ""
        total_steps = agent_response.get("total_steps", 0)
        thoughts = agent_response.get("thoughts_history") or ""
        messages_history = agent_response.get("messages_history") or ""

        full_reasoning_context = f"PROCES MYŚLOWY AGENTA:\n{thoughts}\n\nOSTATECZNA ODPOWIEDŹ:\n{text}"
        score = calibrated_reasoning(question, correct_answer, full_reasoning_context)

        is_answer_correct = float(score.is_answer_correct)
        is_reasoning_correct = float(score.is_reasoning_correct)

        facts = score.extracted_facts
        fact_accuracy = sum(f.is_true for f in facts) / len(facts) if facts else 0.0

        all_facts = [
            {
                "statement": f.statement,
                "is_true": f.is_true,
                "error_explanation": f.error_explanation
            }
            for f in score.extracted_facts
        ]

        record = {
            "question": question,
            "correct_answer": correct_answer,
            "response": text,
            "response_time": response_time,
            "used_words": count_words(text),
            "total_steps": total_steps,
            "thoughts": thoughts,
            "messages_history": messages_history,
            "is_answer_correct": is_answer_correct,
            "is_reasoning_correct": is_reasoning_correct,
            "fact_accuracy": fact_accuracy,
            "all_facts": all_facts,
        }

        print(f"Pytanie: {i} | Kroki: {total_steps} | Poprawna: {bool(is_answer_correct)} | Reasoning: {bool(is_reasoning_correct)} | Facts: {fact_accuracy}")
        print(text)
        print()

    except requests.exceptions.Timeout:
        print(f"⚠️  Pytanie {i}: timeout agenta — pomijam.")
        record = {
            "question": question,
            "correct_answer": correct_answer,
            "response": None,
            "response_time": None,
            "used_words": 0,
            "total_steps": None,
            "thoughts": "",
            "messages_history": "",
            "agent_no_answer": True,
            "is_answer_correct": 0.0,
            "is_reasoning_correct": 0.0,
            "fact_accuracy": 0.0,
            "all_facts": [],
            "error": "timeout",
        }

    except Exception as e:
        print(f"❌ Pytanie {i}: błąd — {e}")
        record = {
            "question": question,
            "correct_answer": correct_answer,
            "response": None,
            "response_time": None,
            "used_words": 0,
            "total_steps": None,
            "thoughts": "",
            "messages_history": "",
            "agent_no_answer": True,
            "is_answer_correct": 0.0,
            "is_reasoning_correct": 0.0,
            "fact_accuracy": 0.0,
            "all_facts": [],
            "error": str(e),
        }

    save_result(record)


Pytanie: 305 | Kroki: 2 | Poprawna: False | Reasoning: False | Facts: 1.0
<fakty_z_bazy>
    Przepraszam, ale w dostępnej bazie wiedzy medycznej nie znalazłem zweryfikowanych informacji na ten temat.
  </fakty_z_bazy>
  <zastrzezenie>
    Informacja wygenerowana na podstawie bazy wiedzy. Nie zastępuje profesjonalnej porady lekarskiej.
  </zastrzezenie>

Pytanie: 306 | Kroki: 2 | Poprawna: False | Reasoning: False | Facts: 1.0
<fakty_z_bazy>
    Przepraszam, ale w dostępnej bazie wiedzy medycznej nie znalazłem zweryfikowanych informacji na ten temat.
  </fakty_z_bazy>
  <zastrzezenie>
    Informacja wygenerowana na podstawie bazy wiedzy. Nie zastępuje profesjonalnej porady lekarskiej.
  </zastrzezenie>

Pytanie: 307 | Kroki: 3 | Poprawna: True | Reasoning: True | Facts: 1.0
<fakty_z_bazy>
    Na podstawie dostępnej bazy wiedzy medycznej, główną przyczyną zgonów osób w wieku podeszłym (65+) w Polsce są choroby układu krążenia. Częstość występowania tych chorób rośnie gwałtownie z wiekiem

# Load & analyze results

In [3]:
rows = []
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except Exception:
            print(f"Pominięto uszkodzoną linię: {line[:80]}")

df = pd.DataFrame(rows)
print(f"Załadowano {len(df)} wyników.")
df.head()

Załadowano 400 wyników.


,question,correct_answer,response,response_time,used_words,total_steps,thoughts,messages_history,is_answer_correct,is_reasoning_correct,fact_accuracy,all_facts,agent_no_answer,error
0,Do nowotworów germinalnych jajnika nie należy:...,D,"<fakty_z_bazy>\n Przepraszam, ale w dostępn...",79.540820,40,2.0,[ANALIZA FRAGMENTÓW]: Brak pobranych fragmentó...,"[{'type': 'ai', 'content': '<analiza> <podmi...",0.0,1.0,1.0,[{'statement': 'W dostępnej bazie wiedzy medyc...,NaN,NaN
1,Podstawowym lekiem stosowanym w POChP są gliko...,E,"<fakty_z_bazy>\n Przepraszam, ale w dostępn...",33.177961,29,1.0,[ANALIZA KONTEKSTU]: Nie otrzymałem żadnego py...,"[{'type': 'ai', 'content': 'Niestety, nie otrz...",0.0,0.0,0.0,[{'statement': 'Nie otrzymałem żadnego pytania...,NaN,NaN
2,Niewydolność cieśniowo-szyjkowa w wywiadzie je...,A,<fakty_z_bazy>\n Na podstawie dostępnej baz...,194.972583,227,4.0,[ZWALIDOWANE FAKTY]:\n- Wskazania do założenia...,"[{'type': 'ai', 'content': '<analiza> <podmi...",1.0,1.0,1.0,[{'statement': 'Wskazania do założenia szwu ok...,NaN,NaN
3,Nazwę „schizofrenia” wprowadził:\nA. Kraepelin...,E,"<fakty_z_bazy>\n Przepraszam, ale w dostępn...",225.734610,36,2.0,[ANALIZA FRAGMENTÓW]: Brak pobranych fragmentó...,"[{'type': 'ai', 'content': '<analiza> <podmi...",0.0,1.0,1.0,[{'statement': 'Dotychczasowe wyniki wyszukiwa...,NaN,NaN
4,W ciągu ostatniego roku zakażenie Legionella p...,A,"<fakty_z_bazy>\n Przepraszam, ale w dostępn...",100.352831,32,3.0,[ANALIZA FRAGMENTÓW]: Brak pobranych fragmentó...,"[{'type': 'ai', 'content': '<analiza> <podmi...",0.0,0.0,1.0,"[{'statement': 'Brak pobranych fragmentów, roz...",NaN,NaN


In [5]:
# -------------------------------------------------------
# Podsumowanie wyników agenta
# -------------------------------------------------------
total = len(df)
errors = df["error"].notna().sum() if "error" in df.columns else 0
valid = total - errors

print("=" * 50)
print("PODSUMOWANIE EWALUACJI AGENTA MEDYCZNEGO")
print("=" * 50)
print(f"Łączna liczba pytań:              {total}")
print(f"Błędy / timeouty:                 {errors}")
print(f"Przetworzone poprawnie:           {valid}")
print()
print(f"Accuracy (is_answer_correct):     {df['is_answer_correct'].mean():.1%}")
print(f"Reasoning correct:                {df['is_reasoning_correct'].mean():.1%}")
print(f"Fact accuracy (avg):              {df['fact_accuracy'].mean():.1%}")
print()
print(f"Agent no-answer rate:             {df['agent_no_answer'].mean():.1%}")
print(f"Avg response time (s):            {df['response_time'].mean():.1f}")
print(f"Avg steps per question:           {df['total_steps'].mean():.1f}")
print(f"Avg words in response:            {df['used_words'].mean():.0f}")
print("=" * 50)

PODSUMOWANIE EWALUACJI AGENTA MEDYCZNEGO
Łączna liczba pytań:              400
Błędy / timeouty:                 3
Przetworzone poprawnie:           397

Accuracy (is_answer_correct):     35.8%
Reasoning correct:                60.2%
Fact accuracy (avg):              85.1%

Agent no-answer rate:             100.0%
Avg response time (s):            158.6
Avg steps per question:           3.5
Avg words in response:            61


In [6]:
# Accuracy vs. knowledge_base_hit
# Sprawdza, czy znalezienie danych w bazie faktycznie poprawia odpowiedź
if "knowledge_base_hit" in df.columns:
    print("Accuracy w zależności od trafienia do bazy wiedzy:")
    print(df.groupby("knowledge_base_hit")["is_answer_correct"].mean().rename("accuracy").to_string())
    print()
    print("Liczba pytań w każdej grupie:")
    print(df["knowledge_base_hit"].value_counts().to_string())

In [ ]:
# Błędne odpowiedzi do analizy
wrong = df[df["is_answer_correct"] == 0.0][["question", "correct_answer", "response", "total_steps", "knowledge_base_hit"]]
print(f"Błędne odpowiedzi: {len(wrong)}")
wrong.head(10)